# Archived-result reproduction
Data/results are not uploaded to GitHub. See [download and preparation instructions](../DATA.md). Raw source data alone do not contain the derived alignment results used here. Set `RAFTUP_EXPERIMENT_ROOT` to your revision archive root. These cells recompute plots and evaluations from saved results, not model training.


# HT206B1: histology, tumour purity and top-five alignment

从已有结果重绘；不训练、不重新配准。先运行路径和公共数据 cell，再分别运行下面三个绘图 cell；每张图可以独立重画。

输入根目录在第一个代码 cell 的 `ROOT`；所有新结果写入本文件夹的 `outputs/03_HT206B1_nonbrain/panels/`，不覆盖原始结果。主要绘图代码已复制进本 notebook，可直接修改。

原始文件：
- `<revision_archive>/all_exp/non_brain/HT206B1/publication_figures/06_HT206B1_publication_figures.ipynb`
- `<revision_archive>/all_exp/non_brain/HT206B1/publication_figures/build_figures.py`
- `<revision_archive>/all_exp/non_brain/HT206B1/publication_figures/combine_figures.py`

论文对应图片：`nonbrain.png`。

三张图均放大字体、移除字母标注，并以零外围留白导出 PDF、SVG、600-dpi PNG 和预览图。


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/raftup_reviewer_figures_mpl')
# Input project root. Change only this path when moving the project.
ROOT = Path(os.environ.get('RAFTUP_EXPERIMENT_ROOT', './revision_data')).expanduser().resolve()
NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', './outputs')).expanduser().resolve()
OUT = RESULTS_DIR / '03_HT206B1_nonbrain' / 'panels'
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Missing input project: {ROOT}'
print('Input project:', ROOT)
print('Outputs:', OUT)



In [ ]:
"""Publication figures from archived coordinates; never trains or refits alignment."""
from pathlib import Path
import os,json,hashlib
os.environ.setdefault('MPLCONFIGDIR','/tmp/ht206_publication_mpl')
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from scipy.ndimage import map_coordinates
DATA = ROOT / 'all_exp/non_brain/HT206B1'
UNIFIED = True  # Same MNN + RANSAC for PRECAST, DeepST and SPIRAL.
os.environ.pop('HT206_FIGURES', None)
SUITE=DATA/'method_comparison'; COMP=SUITE/'comparison'; OLD=DATA/'biological_validation_fc020'; Q=DATA/'quantitative_validation_fc020'
METRIC_COMP=COMP/'unified_mnn_ransac' if UNIFIED else COMP
PAIRS=['U2__U3','U3__U4','U4__U5']; PURPLE='#7030A0'; GREEN='#009E73'; INK='#242424'
plt.rcParams.update({'font.family':'Arial','font.size':7,'axes.titlesize':7,'axes.labelsize':7,'xtick.labelsize':6,'ytick.labelsize':7,'legend.fontsize':6,'axes.linewidth':1,'lines.linewidth':1,'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none','savefig.facecolor':'white','text.color':INK,'axes.labelcolor':INK,'axes.edgecolor':INK})
SOURCES={}
def track(p):
 p=Path(p);SOURCES[str(p)]=hashlib.sha256(p.read_bytes()).hexdigest();return p

# Enlarge all text consistently; export without outer padding.
FONT_SCALE = 1.8
EXPORT_DPI = 600
FIGURE_SIZES = {}
def save(fig,name):
 from matplotlib.text import Text
 from IPython.display import display, Image as DisplayImage
 for artist in fig.findobj(match=Text):
  artist.set_fontsize(artist.get_fontsize()*FONT_SCALE)
 FIGURE_SIZES[name] = list(fig.get_size_inches())
 for ext in ['pdf','svg','png']:
  fig.savefig(OUT/f'{name}.{ext}',dpi=EXPORT_DPI,bbox_inches='tight',pad_inches=0)
 preview=OUT/f'{name}_preview.png'
 fig.savefig(preview,dpi=180,bbox_inches='tight',pad_inches=0)
 plt.close(fig)
 display(DisplayImage(filename=str(preview)))
 print(OUT/f'{name}.pdf')

def label(ax,s):
 # No letter annotations in the exported figures.
 pass
def scalebar(ax,x,y,length=1000):
 ax.plot([x,x+length],[y,y],color=INK,lw=1.5,solid_capstyle='butt');ax.text(x+length/2,y-90,'1 mm',ha='center',va='bottom',fontsize=6)
def spatial(ax,lim):
 ax.set_xlim(lim[0],lim[1]);ax.set_ylim(lim[3],lim[2]);ax.set_aspect('equal');ax.axis('off')

metrics=pd.read_csv(track(METRIC_COMP/'purity_metrics.csv'))
assert not metrics.duplicated(['method','pair']).any()
assert all(set(z.pair)==set(PAIRS) for _,z in metrics.groupby('method'))
means=metrics.groupby('method')[['purity_spearman','purity_MAE']].mean().sort_values('purity_spearman',ascending=False)
archived=pd.read_csv(track(METRIC_COMP/'all_MNN_mean_metrics.csv')).set_index('method')
assert np.allclose(means.sort_index(),archived.loc[sorted(means.index),means.columns])
TOP=means.index[:5].tolist()
means.to_csv(OUT/'source_data_mean_metrics.csv');metrics.to_csv(OUT/'source_data_pair_metrics.csv',index=False)
# Coordinate/metric identity audit using the unified MNN + RANSAC coordinates.
coords={};links={};audits=[]
base=pd.read_csv(track(OLD/'aligned_spots_with_purity.csv.gz'))
coords['RAFT-UP']=pd.concat([base[base.section.isin(p.split('__'))].assign(pair=p) for p in PAIRS],ignore_index=True)
purity={s:pd.read_csv(track(OLD/f'{s}_metadata.csv')).set_index('barcode').Tumor_purity_estimate for s in ['U2','U3','U4','U5']}
for m in means.index:
 if m!='RAFT-UP':
  folder=(METRIC_COMP/m if UNIFIED and m in ['SPIRAL','PRECAST','DeepST'] else SUITE/'results'/m); state=json.loads(track(folder/'run.json').read_text());p=track(folder/'pair_coordinates.csv.gz')
  assert state['status']=='completed' and hashlib.sha256(p.read_bytes()).hexdigest()==state['coordinates_sha256']
  coords[m]=pd.read_csv(p)
 parts=[]
 for pair in PAIRS:
  a,b=pair.split('__');d=coords[m];A=d[(d.pair==pair)&(d.section==a)];B=d[(d.pair==pair)&(d.section==b)]
  x=A[['x_um','y_um']].to_numpy();y=B[['x_um','y_um']].to_numpy();dist,j=cKDTree(y).query(x);rev=cKDTree(x).query(y)[1]
  i=np.flatnonzero((rev[j]==np.arange(len(x)))&(dist<=100));jj=j[i]
  v=purity[a].loc[A.barcode.iloc[i]].to_numpy();w=purity[b].loc[B.barcode.iloc[jj]].to_numpy()
  rho=float(spearmanr(v,w)[0]);mae=float(np.mean(np.abs(v-w)));old=metrics[(metrics.method==m)&(metrics.pair==pair)].iloc[0]
  assert len(i)==old.n_pairs and np.allclose([rho,mae],[old.purity_spearman,old.purity_MAE],atol=1e-12), (m,pair,rho,mae,old)
  audits.append(dict(method=m,pair=pair,n_pairs=len(i),purity_spearman=rho,purity_MAE=mae,verified=True))
  parts.append(pd.DataFrame(dict(pair=pair,barcode_A=A.barcode.iloc[i].to_numpy(),barcode_B=B.barcode.iloc[jj].to_numpy(),purity_A=v,purity_B=w,x_A=x[i,0],y_A=x[i,1],x_B=y[jj,0],y_B=y[jj,1],distance_um=dist[i])))
 links[m]=pd.concat(parts,ignore_index=True)
pd.DataFrame(audits).to_csv(OUT/'coordinate_metric_audit.csv',index=False)
print('Verified all 27 method/pair metrics from exact plotted coordinate versions.',flush=True)


labels={'DeepST':'DeepST + rigid','PRECAST':'PRECAST + rigid'}
if UNIFIED:labels['SPIRAL']='SPIRAL + rigid'


## 组织学与 tumour-purity 说明图


In [ ]:
# Figure 2: actual image-derived masks and actual RAFT-UP MNN purity pairs.
# U3/U4 is the middle adjacent pair, chosen before inspecting its score.
a,b='U3','U4';pair=a+'__'+b
cal=pd.read_csv(track(DATA/'raftup_manual_fc020_v1/calibration.csv')).set_index('section')
ims={};fac={};mask={}
for s in [a,b]:
 prefix=DATA/f'HT206B1-S1Fc1{s}Z1B1'
 ims[s]=np.asarray(Image.open(track(Path(str(prefix)+'-tissue_lowres_image.png'))).convert('RGB'))
 sc=json.loads(track(Path(str(prefix)+'-scalefactors_json.json')).read_text());fac[s]=100/cal.loc[s,'pixels_per_100um']/sc['tissue_lowres_scalef']
 mask[s]=np.asarray(Image.open(track(Q/f'{s}_dark_stain_proxy_mask.png')))>0
tr=json.loads(track(DATA/'raftup_manual_fc020_v1'/pair/'rigid_b_to_a.json').read_text());R=np.array(tr['R_row_vector']);t=np.array(tr['t'])
# Independent pair frame for images, same archived transform as validation.
def corners(s):
 hh,ww=mask[s].shape;return np.array([[0,0],[ww,0],[0,hh],[ww,hh]])*fac[s]
raw={}
for section in [a,b]:
 pp=track(DATA/f'HT206B1-S1Fc1{section}Z1B1-tissue_positions_list.csv')
 pos=pd.read_csv(pp,header=None,names=['barcode','in_tissue','row','col','y','x']).set_index('barcode')
 raw[section]=pos.loc[purity[section].index,['x','y']].to_numpy()*100/cal.loc[section,'pixels_per_100um']
trs=[(R,t),(np.eye(2),raw[a].mean(0)-raw[b].mean(0))]
center=(raw[b]@R+t).mean(0)
for deg in [-20,-10,10,20]:
 angle=np.deg2rad(deg);rot=np.array([[np.cos(angle),np.sin(angle)],[-np.sin(angle),np.cos(angle)]])
 trs.append((R@rot,(t-center)@rot+center))
for delta in [[500,0],[-500,0],[0,500],[0,-500]]:trs.append((R,t+delta))
cc=np.vstack([corners(a)]+[corners(b)@r+tt for r,tt in trs]);lo=cc.min(0)-40;hi=cc.max(0)+40;xx,yy=np.meshgrid(np.arange(lo[0],hi[0]+20,20),np.arange(lo[1],hi[1]+20,20));grid=np.stack([xx,yy],-1)
warps=[]
for s,r,tt in [(a,np.eye(2),np.zeros(2)),(b,R,t)]:
 pix=(grid-tt)@r.T/fac[s];warps.append(map_coordinates(mask[s].astype(float),[pix[:,:,1],pix[:,:,0]],order=0,mode='constant',cval=0)>.5)
dice=2*np.logical_and(*warps).sum()/sum(z.sum() for z in warps)
expected=pd.read_csv(track(Q/'region_overlap_boundary_metrics.csv'));expected=expected[(expected.pair==pair)&(expected.region=='dark_stain_proxy')&(expected.method=='RAFT-UP')].iloc[0].dice
# Same grid as the archived 20-um mask evaluation.
assert abs(dice-expected)<1e-12
fig,axes=plt.subplots(2,3,figsize=(12,11.6));fig.subplots_adjust(left=.06,right=.97,bottom=.21,top=.81,hspace=.6,wspace=.30)
fig.text(.5,.968,'Histology and tumour-purity by RAFT-UP',ha='center',fontsize=9,fontweight='bold')
fig.text(.06,.875,'Dark-stain proxy: H&E → hematoxylin-rich mask → overlap',fontweight='bold',fontsize=8)
for j,s in enumerate([a,b]):
 ax=axes[0,j];hh,ww=mask[s].shape
 ax.imshow(ims[s],extent=[0,ww*fac[s],hh*fac[s],0],interpolation='nearest')
 tint=np.zeros((*mask[s].shape,4));tint[mask[s]]=to_rgba('#0072B2' if j==0 else '#E69F00',.62)
 ax.imshow(tint,extent=[0,ww*fac[s],hh*fac[s],0],interpolation='nearest')
 ax.set_title(s+': thresholded hematoxylin',pad=5);ax.axis('off');label(ax,chr(97+j));scalebar(ax,400,hh*fac[s]-250)
ax=axes[0,2];rgba=np.ones((*warps[0].shape,4));rgba[warps[0]]=to_rgba('#0072B2',.62);rgba[warps[1]]=to_rgba('#E69F00',.62);rgba[warps[0]&warps[1]]=to_rgba('#252525',1)
ax.imshow(rgba,extent=[lo[0],xx[0,-1],yy[-1,0],lo[1]],interpolation='nearest');ax.set_title('Aligned masks · Dice = '+f'{dice:.3f}');ax.axis('off');label(ax,'c')
scalebar(ax,lo[0]+400,yy[-1,0]-250)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#0072B2',label='U3'),Patch(facecolor='#E69F00',label='U4'),Patch(facecolor='#252525',label='Overlap')],loc='upper center',bbox_to_anchor=(.5,1.0),ncol=3,frameon=False,fontsize=6,handlelength=1,columnspacing=.8,handletextpad=.4,borderaxespad=0)
fig.text(.06,.505,'Spatial MNN pairs → paired tumour-purity estimates → concordance',fontweight='bold',fontsize=8)
z=links['RAFT-UP'];z=z[z.pair==pair];r=float(spearmanr(z.purity_A,z.purity_B)[0]);mae=float(np.abs(z.purity_A-z.purity_B).mean())
# Match global spatial frame of the validated purity estimates (not image pair frame).
allxy=np.vstack([z[['x_A','y_A']],z[['x_B','y_B']]]);lo2=allxy.min(0)-300;hi2=allxy.max(0)+300;lim=[lo2[0],hi2[0],lo2[1],hi2[1]]
for j,suffix in enumerate(['A','B']):
 ax=axes[1,j];scat=ax.scatter(z['x_'+suffix],z['y_'+suffix],c=z['purity_'+suffix],cmap='viridis',vmin=0,vmax=1,s=4.0,linewidths=0)
 spatial(ax,lim);ax.set_title([a+': matched spots',b+': matched spots (aligned)'][j]);label(ax,chr(100+j));scalebar(ax,lim[0]+250,lim[3]-200)
cax=fig.add_axes([.095,.123,.48,.012]);cb=fig.colorbar(scat,cax=cax,orientation='horizontal',ticks=[0,.5,1]);cb.set_label('Estimated tumour purity',labelpad=1)
ax=axes[1,2];hb=ax.hexbin(z.purity_A,z.purity_B,gridsize=27,mincnt=1,cmap='magma',linewidths=0);ax.plot([0,1],[0,1],ls='--',color='#737373',lw=1)
ax.set(xlim=(0,1),ylim=(0,1),xlabel='U3 tumour purity',ylabel='U4 tumour purity',xticks=[0,.5,1],yticks=[0,.5,1]);ax.set_aspect('equal');ax.spines[['top','right']].set_visible(False)
ax.set_title(f'{len(z):,} spatial MNN pairs\nρ = {r:.3f}; MAE = {mae:.3f}');label(ax,'f')
cax=fig.add_axes([.746,.123,.20,.012]);cb=fig.colorbar(hb,cax=cax,orientation='horizontal');cb.set_label('Pairs per hexagon',labelpad=1)
save(fig,'figure_2_metric_explanation');z.to_csv(OUT/'source_data_explanation_MNN_pairs.csv',index=False)



## 各方法的 tumour-purity 指标比较


In [ ]:
# Figure 1: equal-weight means, with all three section-pair observations.
fig,axes=plt.subplots(1,2,figsize=(12,7.4));fig.subplots_adjust(left=.18,right=.975,bottom=.19,top=.78,wspace=.45)
fig.text(.5,.957,'Tumour-purity concordance in serial breast cancer sections',ha='center',fontsize=9,fontweight='bold')
fig.text(.5,.909,'HT206B1 · Visium · Sections U2–U5 · All spatial MNN pairs ≤100 μm',ha='center',fontsize=7)
for k,(ax,col,title) in enumerate(zip(axes,['purity_spearman','purity_MAE'],['Tumour-purity rank correlation','Absolute tumour-purity difference'])):
 for i,m in enumerate(means.index):
  color=PURPLE if m=='RAFT-UP' else '#8D9BA9'
  ax.barh(i,means.loc[m,col],height=.62,color=color,zorder=2)
  for q,(pair,marker,delta) in enumerate(zip(PAIRS,['o','s','^'],[-.16,0,.16])):
   value=metrics[(metrics.method==m)&(metrics.pair==pair)][col].iloc[0]
   ax.scatter(value,i+delta,marker=marker,s=28,facecolor='white',edgecolor=INK,linewidth=1,zorder=3)
  ax.text(.985,i,f'{means.loc[m,col]:.4f}',transform=ax.get_yaxis_transform(),ha='right',va='center',fontsize=6)
 ax.set_yticks(range(len(means)),[labels.get(m,m) for m in means.index]);ax.invert_yaxis()
 for t,m in zip(ax.get_yticklabels(),means.index):
  if m=='RAFT-UP':t.set_fontweight('bold');t.set_color(PURPLE)
 ax.set_xlim((0,.86) if k==0 else (0,.19));ax.set_title(title,pad=9)
 ax.set_xlabel('Spearman ρ (higher is better)' if k==0 else 'Mean absolute error (lower is better)')
 ax.spines[['top','right']].set_visible(False);ax.tick_params(axis='y',length=0);ax.set_axisbelow(True);ax.xaxis.grid(True,color='#E6E6E6',lw=1)
 label(ax,chr(97+k))
handles=[Line2D([],[],marker=mark,ls='',markerfacecolor='white',markeredgecolor=INK,label=p.replace('__','–'),markersize=6) for p,mark in zip(PAIRS,['o','s','^'])]
fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(.56,.040),ncol=3,frameon=False)
save(fig,'figure_1_purity_concordance')



## Top-five 空间对齐图


In [ ]:
# Figure 3: same five selected methods; SPIRAL uses requested rigid reconstruction.
plot_coords=dict(coords);plot_metrics=metrics.copy();plot_labels=dict(labels)
if not UNIFIED:
 rigid_folder=COMP/'learned_similarity_rigid/SPIRAL_expdist_rigid'
 rigid_state=json.loads(track(rigid_folder/'run.json').read_text())
 rigid_path=track(rigid_folder/'pair_coordinates.csv.gz')
 assert rigid_state['status']=='completed'
 assert hashlib.sha256(rigid_path.read_bytes()).hexdigest()==rigid_state['coordinates_sha256']
 plot_coords['SPIRAL']=pd.read_csv(rigid_path)
 rigid_metrics=pd.read_csv(track(rigid_folder/'purity_metrics.csv'))
 for pair in PAIRS:
  a,b=pair.split('__');d=plot_coords['SPIRAL'];A=d[(d.pair==pair)&(d.section==a)];B=d[(d.pair==pair)&(d.section==b)]
  x=A[['x_um','y_um']].to_numpy();y=B[['x_um','y_um']].to_numpy()
  dist,j=cKDTree(y).query(x);rev=cKDTree(x).query(y)[1];i=np.flatnonzero((rev[j]==np.arange(len(x)))&(dist<=100))
  v=purity[a].loc[A.barcode.iloc[i]].to_numpy();w=purity[b].loc[B.barcode.iloc[j[i]]].to_numpy()
  row=rigid_metrics[rigid_metrics.pair==pair].iloc[0]
  assert len(i)==row.n_pairs and np.allclose([spearmanr(v,w)[0],np.abs(v-w).mean()],[row.purity_spearman,row.purity_MAE],atol=1e-12)
 rigid_metrics=rigid_metrics.assign(method='SPIRAL')
 plot_metrics=pd.concat([metrics[metrics.method!='SPIRAL'],rigid_metrics],ignore_index=True)
 plot_means=plot_metrics.groupby('method')[['purity_spearman','purity_MAE']].mean()
 plot_labels['SPIRAL']='SPIRAL + rigid'
 plot_metrics[plot_metrics.method.isin(TOP)].assign(coordinate_version=lambda df:np.where(df.method=='SPIRAL','exp(-distance) rigid','original comparison')).to_csv(OUT/'source_data_alignment_display_metrics.csv',index=False)
else:
 plot_means=means
 plot_metrics[plot_metrics.method.isin(TOP)].to_csv(OUT/'source_data_alignment_display_metrics.csv',index=False)
fig,axes=plt.subplots(5,3,figsize=(12,16));fig.subplots_adjust(left=.205,right=.98,top=.895,bottom=.045,wspace=.06,hspace=.14)
fig.text(.5,.977,'Spatial alignment of serial breast cancer sections (HT206B1)',ha='center',fontweight='bold',fontsize=9)
fig.text(.5,.955,('Top five by mean purity Spearman ρ · Embedding methods use shared MNN + RANSAC' if UNIFIED else 'Five methods selected from the purity ranking; SPIRAL shown with rigid reconstruction'),ha='center',fontsize=7)
fig.legend(handles=[Line2D([],[],marker='o',color=PURPLE,ls='',label='Earlier section',markersize=6),Line2D([],[],marker='^',color=GREEN,ls='',label='Later section (aligned)',markersize=6)],loc='upper center',bbox_to_anchor=(.61,.941),frameon=False,ncol=2)
# One physical scale for all 15 panels; centre-only view limits preserve coordinates.
span=0
for m in TOP:
 for pair in PAIRS:
  d=plot_coords[m];pts=d[d.pair==pair][['x_um','y_um']].to_numpy();span=max(span,np.ptp(pts,axis=0).max())
span=np.ceil((span+650)/100)*100
for i,m in enumerate(TOP):
 for j,pair in enumerate(PAIRS):
  ax=axes[i,j];a,b=pair.split('__');d=plot_coords[m];d=d[d.pair==pair];pts=d[['x_um','y_um']].to_numpy();center=(pts.min(0)+pts.max(0))/2
  lim=[center[0]-span/2,center[0]+span/2,center[1]-span/2,center[1]+span/2]
  for s,col,mark in [(a,PURPLE,'o'),(b,GREEN,'^')]:
   v=d[d.section==s];ax.scatter(v.x_um,v.y_um,s=3.1,c=col,marker=mark,alpha=.70,linewidths=0)
  spatial(ax,lim);scalebar(ax,lim[0]+300,lim[3]-230)
  row=plot_metrics[(plot_metrics.method==m)&(plot_metrics.pair==pair)].iloc[0]
  ax.text(.5,-.085,f'ρ = {row.purity_spearman:.3f}   MAE = {row.purity_MAE:.3f}',transform=ax.transAxes,ha='center',fontsize=6)
  if i==0:ax.set_title(pair.replace('__','–'),fontweight='bold',pad=5)
  if j==0:
   ax.text(-.12,.62,plot_labels.get(m,m),transform=ax.transAxes,ha='right',va='center',fontweight='bold',fontsize=7,color=PURPLE if m=='RAFT-UP' else INK)
   ax.text(-.12,.40,f'Mean ρ = {plot_means.loc[m,"purity_spearman"]:.3f}',transform=ax.transAxes,ha='right',va='center',fontsize=6,linespacing=1.6)
save(fig,'figure_3_top5_alignment')
if (NOTEBOOK_DIR / '03_HT206B1_nonbrain.ipynb').is_file():
 track(NOTEBOOK_DIR / '03_HT206B1_nonbrain.ipynb')
(OUT/'provenance.json').write_text(json.dumps({'sources_sha256':SOURCES,'top5':TOP,'ranking':'equal-weight mean Spearman across three adjacent pairs','spiral_version':('shared L2 embedding MNN + RANSAC in both metric and coordinate figures' if UNIFIED else {'figure_1':'native weighted_mean alpha=.5','figure_3':'exp(-distance) rigid; per-panel and mean metrics from rigid coordinates'}),'figure_sizes_inches':FIGURE_SIZES,'export_dpi':EXPORT_DPI,'font_scale':FONT_SCALE,'outer_padding_inches':0,'alignment_rerun':False},indent=2))
print('Saved three figures in PDF, SVG, 600-dpi PNG and preview PNG.',flush=True)
